#### Validando se o arquivo novo perde ou não dados e se repete as linhas:

In [ ]:
# === Importando Bibliotecas e consumindo a Base de Dados original ===
from openpyxl.styles import PatternFill, Font
from openpyxl import load_workbook
import pandas as pd

# === CONFIGURAÇÃO ===
arquivo_excel = r".\xlsx\Departamentos-Afastados - Novo.xlsx"                               # Primeiro arquivo...
arquivo_comparacao = r".\xlsx\Departamentos-Afastados - Antigo.xlsx"                        # Segundo arquivo...
saida = r"..\..\Base de Dados Original\xlsx\Departamentos-Afastados\Departamentos-Afastados.xlsx"    # Arquivo final...

# === ABRIR O ARQUIVO ORIGINAL (para preservar layout) ===
wb = load_workbook(arquivo_excel)

# === LEITURA DAS PLANILHAS ===
pagina1 = 0
pagina2 = 1
pagina3 = 2
pagina4 = 3

df1 = pd.read_excel(arquivo_excel, sheet_name=pagina1)
df2 = pd.read_excel(arquivo_excel, sheet_name=pagina2)
df3 = pd.read_excel(arquivo_excel, sheet_name=pagina3)
df4 = pd.read_excel(arquivo_excel, sheet_name=pagina4)

In [ ]:
# === FUNÇÃO DE LIMPEZA ===
def limpar(df):
    df.columns = df.columns.str.strip().str.lower()
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype(str).str.strip().str.lower()
    return df

# === APLICAR LIMPEZA ===
df1, df2, df3, df4 = map(limpar, [df1, df2, df3, df4])

# === FUNÇÃO PARA PREPARAR DATAS ===
def preparar_datas(df, nome_coluna):
    if nome_coluna.lower() in df.columns:
        df[nome_coluna.lower()] = df[nome_coluna.lower()].astype(str)
        df[nome_coluna.lower()] = df[nome_coluna.lower()].str.replace('.', '/', regex=False)
        df[nome_coluna.lower()] = pd.to_datetime(df[nome_coluna.lower()], errors='coerce', dayfirst=True)
        df['ano'] = df[nome_coluna.lower()].dt.year
        df['mes'] = df[nome_coluna.lower()].dt.month
    return df

# === PREPARAR DATAS NAS PÁGINAS ===
df1 = preparar_datas(df1, 'Data_Admissão')
df3 = preparar_datas(df3, 'Data_Admissão')
df2 = preparar_datas(df2, 'Data_Dem')
df4 = preparar_datas(df4, 'Data_Dem')

# === FUNÇÃO DE COMPARAÇÃO ===
def comparar_paginas(df_origem, df_destino, nome_origem, nome_destino, coluna_data):
    # Selecionar períodos comuns
    periodos_comuns = set(zip(df_origem['ano'], df_origem['mes'])) & set(zip(df_destino['ano'], df_destino['mes']))
    df_origem_periodo = df_origem[df_origem[['ano', 'mes']].apply(tuple, axis=1).isin(periodos_comuns)]

    # Linhas do origem que não estão no destino
    faltando = df_origem_periodo.merge(df_destino, how='left', indicator=True)
    faltando = faltando[faltando['_merge'] == 'left_only'].drop(columns=['_merge'])
    faltando['pagina'] = nome_origem

    print(f"\n🔍 {nome_origem} → {nome_destino}")
    if faltando.empty:
        print("✅ Nenhuma linha diferente encontrada no mesmo período.")
    else:
        print(f"⚠️ {len(faltando)} linha(s) novas em {nome_origem} dentro do mesmo período de {nome_destino}.")

    return faltando

In [ ]:
# === ABRIR O ARQUIVO ORIGINAL (mantém formatação e maiúsculas) ===
wb = load_workbook(arquivo_excel)
print("Abas encontradas:", wb.sheetnames)

# === FUNÇÃO PARA LER UMA ABA COMO TEXTO PURO (sem pandas lower) ===
def ler_aba_como_dataframe(nome_aba):
    ws = wb[nome_aba]
    dados = []

    for row in ws.iter_rows(values_only=True):
        dados.append(list(row))

    # Primeira linha são as colunas
    colunas = [str(c) if c is not None else f"Col_{i}" for i, c in enumerate(dados[0])]
    df = pd.DataFrame(dados[1:], columns=colunas)
    return df

# === LER AS 4 ABAS COM openpyxl (preserva conteúdo original) ===
df1 = ler_aba_como_dataframe(wb.sheetnames[0])
df2 = ler_aba_como_dataframe(wb.sheetnames[1])
df3 = ler_aba_como_dataframe(wb.sheetnames[2])
df4 = ler_aba_como_dataframe(wb.sheetnames[3])

# === FUNÇÃO PARA UNIR E ATUALIZAR UMA ABA PRESERVANDO LAYOUT E MAIÚSCULAS ===
def unir_e_atualizar(indice_destino, df_a, df_b, nome_destino):
    nome_aba = wb.sheetnames[indice_destino]
    ws = wb[nome_aba]

    # Mantém a ordem e nomes das colunas da aba de destino
    colunas_destino = list(df_a.columns)
    for col in df_b.columns:
        if col not in colunas_destino:
            colunas_destino.append(col)

    df_a = df_a.reindex(columns=colunas_destino)
    df_b = df_b.reindex(columns=colunas_destino)

    # Marca origem
    df_a["origem"] = "1_ou_2"
    df_b["origem"] = "3_ou_4"

    # União das páginas
    df_unido = pd.concat([df_a, df_b], ignore_index=True)

    # ===============================
    # 🔍 REGRA: MESMO NOME + Nº PESS.
    # ===============================
    if all(col in df_unido.columns for col in ["Nome completo", "Nº pess."]):
        chaves = ["Nome completo", "Nº pess."]

        # Ordena de forma que 3/4 venham depois — e sejam mantidos no drop_duplicates
        df_unido["prioridade"] = df_unido["origem"].apply(lambda x: 1 if x == "3_ou_4" else 0)
        df_unido = df_unido.sort_values(by="prioridade", ascending=True)

        # Remove duplicados, mantendo o último (ou seja, o da 3/4)
        antes = len(df_unido)
        df_unido = df_unido.drop_duplicates(subset=chaves, keep="last")
        depois = len(df_unido)

        print(f"🧹 {antes - depois} duplicada(s) removida(s) em {nome_destino} — mantendo versão das páginas 3/4.")

        # Remove colunas auxiliares
        df_unido = df_unido.drop(columns=["origem", "prioridade"])

    else:
        df_unido = df_unido.drop_duplicates()
        print(f"⚠️ Colunas esperadas ('Nome completo', 'Nº pess.') não encontradas em {nome_destino}.")

    # ===============================
    # ✍️ ESCREVER DE VOLTA NA PLANILHA
    # ===============================
    print(f"📄 {nome_destino}: {len(df_unido)} linhas únicas (mantendo dados das páginas 3/4).")

    # Limpa linhas antigas (mantendo cabeçalhos e estilos)
    ws.delete_rows(2, ws.max_row)

    # Escreve célula por célula, preservando formatação e maiúsculas
    for r_idx, row in enumerate(df_unido.itertuples(index=False), start=2):
        for c_idx, valor in enumerate(row, start=1):
            ws.cell(row=r_idx, column=c_idx, value=None if pd.isna(valor) else valor)

# === NOVA FUNÇÃO: REMOVER PESSOAS DA ABA 1 QUE TAMBÉM ESTÃO NA ABA 2 ===
def remover_duplicados_entre_abas(indice_aba1, indice_aba2):
    nome_aba1 = wb.sheetnames[indice_aba1]
    nome_aba2 = wb.sheetnames[indice_aba2]

    print(f"\n🔁 Comparando '{nome_aba1}' x '{nome_aba2}' para remover pessoas duplicadas da primeira aba...")

    ws1 = wb[nome_aba1]

    df1_atual = ler_aba_como_dataframe(nome_aba1)
    df2_atual = ler_aba_como_dataframe(nome_aba2)

    # Garante que as colunas-chave existem nas duas abas
    chaves = ["Nome completo", "Nº pess."]
    if not all(col in df1_atual.columns for col in chaves) or not all(col in df2_atual.columns for col in chaves):
        print("⚠️ Não foi possível remover duplicados entre abas: colunas 'Nome completo' e 'Nº pess.' não estão presentes nas duas abas.")
        return

    # Cria chave padronizada (ignorando maiúsculas/minúsculas e espaços)
    def criar_chave(df):
        return (
            df["Nome completo"].astype(str).str.strip().str.lower()
            + "||"
            + df["Nº pess."].astype(str).str.strip().str.lower()
        )

    df1_atual["_chave"] = criar_chave(df1_atual)
    df2_atual["_chave"] = criar_chave(df2_atual)

    chaves_segunda = set(df2_atual["_chave"].dropna())

    antes = len(df1_atual)
    df1_filtrado = df1_atual[~df1_atual["_chave"].isin(chaves_segunda)].drop(columns=["_chave"])
    removidas = antes - len(df1_filtrado)

    # Remove coluna auxiliar da segunda aba também (não será reescrita, mas só por segurança)
    df2_atual = df2_atual.drop(columns=["_chave"])

    print(f"🧹 {removidas} linha(s) removida(s) da aba '{nome_aba1}' porque a mesma pessoa está na aba '{nome_aba2}' (mantendo só na segunda).")

    # Reescreve a ABA 1 com as linhas filtradas (mantém cabeçalho original)
    ws1.delete_rows(2, ws1.max_row)
    for r_idx, row in enumerate(df1_filtrado.itertuples(index=False), start=2):
        for c_idx, valor in enumerate(row, start=1):
            ws1.cell(row=r_idx, column=c_idx, value=None if pd.isna(valor) else valor)

# === UNIR Página 1 + Página 3 ===
unir_e_atualizar(0, df1, df3, "Página 1 + Página 3")

# === UNIR Página 2 + Página 4 ===
unir_e_atualizar(1, df2, df4, "Página 2 + Página 4")

# === AGORA COMPARA ABA 1 x ABA 2 E REMOVE DA 1 QUEM JÁ ESTÁ NA 2 ===
remover_duplicados_entre_abas(0, 1)

# === RENOMEAR A SEGUNDA ABA PARA 'Desligados' ===
wb.sheetnames  # garante que a lista está atualizada
segunda_aba = wb.worksheets[1]
segunda_aba.title = "Desligados"
print("📝 Aba renomeada: Página 2 + Página 4 → Desligados")

# === REMOVER ABAS 3 e 4 (deixa só as duas primeiras) ===
while len(wb.sheetnames) > 2:
    nome = wb.sheetnames[-1]
    del wb[nome]
    print(f"🗑️ Aba removida: {nome}")

# === SALVAR O NOVO ARQUIVO ===
wb.save(saida)
print(f"\n✅ Arquivo final salvo preservando maiúsculas, conteúdo e visual em: {saida}")